In [1]:
"""
=============================================================================
  Epoch-Level Filter Calibration Test
  (isolated, standalone -- does NOT touch the preprocessing pipeline)
=============================================================================

Purpose
-------
Before implementing the epoch-level ML-anomaly filter inside
Preprocessing_v2_run_a1b_10s.ipynb (D2 in the improvement checklist), this
script answers the calibration question first: "what overlap threshold
should we use, and how much data would each candidate threshold remove?"

It does this WITHOUT touching preprocessing:
  1. Loads raw (session-cropped) EEG for 3 known temporal_emg subjects
     (VP001, VP006, VP008 -- confirmed in earlier manual review).
  2. Runs the same ML ensemble anomaly detector (Isolation Forest + LOF +
     Elliptic Envelope majority vote) used in the artifact-detection
     script, to get anomaly time windows.
  3. Simulates the EXACT sliding-window scheme used by preprocessing
     (EEG_WINDOW_LEN=10.0s, EEG_STEP=1.0s, per the preprocessing config)
     across the whole session.
  4. For a range of candidate overlap thresholds (10% to 50%), computes
     what fraction of simulated windows would be removed.
  5. Applies the decision rule from the improvement checklist:
        removal_fraction < 30%  -> threshold usable, filter this subject
        removal_fraction > 40%  -> too much data lost, consider full
                                    subject removal instead of filtering

Output
------
  - A printed table: subject x session x threshold -> % windows removed
  - A CSV with the same data
  - A recommendation printed at the end for each subject

Install:
  pip install numpy scipy scikit-learn mne
=============================================================================
"""

import os
import csv
import warnings
import numpy as np
import mne
mne.set_log_level("WARNING")
warnings.filterwarnings("ignore")

from scipy import stats
from scipy.signal import welch, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA

# ══════════════════════════════════════════════════════════════════════════
#  0. Configuration
# ══════════════════════════════════════════════════════════════════════════

PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s-onetime ica"
OUTPUT_DIR           = r"E:\kntu\Thesis\codes\Results\epoch_filter_calibration"

# Subjects confirmed to show the temporal_emg artifact pattern in the
# earlier manual review (VP001, VP006, VP008)
TEST_SUBJECTS = ["VP001", "VP006", "VP008"]
SESSIONS      = [1, 2, 3]

SFREQ = 200.0

# Preprocessing window scheme -- MUST match Preprocessing_v2_run_a1b_10s.ipynb
EEG_WINDOW_LEN = 10.0   # seconds (EEG_WINDOW_LEN in preprocessing config)
EEG_STEP       = 1.0    # seconds (EEG_STEP in preprocessing config)

# Candidate overlap thresholds to test (fraction of the 10s window that
# must overlap with a flagged ML-anomaly epoch for the window to be removed)
CANDIDATE_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50]

CONTAMINATION_ML = 0.10


# ══════════════════════════════════════════════════════════════════════════
#  1. Load raw (session-cropped) EEG -- same method as the artifact script
# ══════════════════════════════════════════════════════════════════════════

def load_session_eeg(subject_id: str, session: int) -> np.ndarray | None:
    fif_path = os.path.join(PROCESSED_DATA_PATH, subject_id,
                            f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(fif_path):
        return None

    raw = mne.io.read_raw_fif(fif_path, preload=False, verbose=False)
    total_dur   = raw.times[-1]
    session_dur = total_dur / 3.0
    t_start = (session - 1) * session_dur
    t_end   = min(session * session_dur, total_dur)

    raw_crop = raw.copy().crop(tmin=t_start, tmax=t_end)
    raw_crop.load_data()
    data = raw_crop.get_data() * 1e6   # Volt -> microvolt
    del raw_crop
    return data.astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════
#  2. ML ensemble anomaly detection (same method as the artifact script)
# ══════════════════════════════════════════════════════════════════════════

def bandpower(x: np.ndarray, sfreq: float, band: tuple) -> float:
    f, pxx = welch(x, fs=sfreq, nperseg=min(512, len(x)))
    idx = (f >= band[0]) & (f <= band[1])
    return float(np.trapz(pxx[idx], f[idx]))


def extract_epoch_features(epoch: np.ndarray, sfreq: float) -> np.ndarray:
    feats = []
    for sig in epoch:
        ch_f = [
            float(np.std(sig)), float(stats.kurtosis(sig)), float(stats.skew(sig)),
            float(np.ptp(sig)), float(np.sqrt(np.mean(sig ** 2))),
            bandpower(sig, sfreq, (0.5, 4)), bandpower(sig, sfreq, (4, 8)),
            bandpower(sig, sfreq, (8, 13)), bandpower(sig, sfreq, (13, 30)),
            bandpower(sig, sfreq, (30, min(90, sfreq / 2 - 1))),
        ]
        feats.extend(ch_f)
    return np.array(feats, dtype=np.float32)


def epoch_signal(data: np.ndarray, sfreq: float, epoch_len=2.0, overlap=0.5):
    step = int(epoch_len * sfreq * (1 - overlap))
    win  = int(epoch_len * sfreq)
    epochs, starts = [], []
    for s in range(0, data.shape[1] - win + 1, step):
        epochs.append(data[:, s:s + win])
        starts.append(s / sfreq)
    return np.array(epochs), np.array(starts)


def get_ml_anomaly_times(data: np.ndarray, sfreq: float) -> list:
    """
    Runs the Isolation Forest + LOF + Elliptic Envelope majority-vote
    ensemble (identical to the artifact-detection script) and returns the
    list of 2-second epoch start times flagged as anomalous by >=2 of 3
    detectors.
    """
    epochs, times = epoch_signal(data, sfreq, epoch_len=2.0, overlap=0.5)
    X = np.array([extract_epoch_features(ep, sfreq) for ep in epochs], dtype=np.float32)
    X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)
    X_sc = StandardScaler().fit_transform(X)

    iso  = IsolationForest(contamination=CONTAMINATION_ML, n_estimators=150, random_state=42)
    p_if = iso.fit_predict(X_sc)

    lof  = LocalOutlierFactor(n_neighbors=min(20, len(X_sc) - 1), contamination=CONTAMINATION_ML)
    p_lof = lof.fit_predict(X_sc)

    try:
        pca_x = PCA(n_components=min(10, X_sc.shape[1], X_sc.shape[0] - 1)).fit_transform(X_sc)
        ee    = EllipticEnvelope(contamination=CONTAMINATION_ML, random_state=42, support_fraction=0.9)
        p_ee  = ee.fit_predict(pca_x)
    except Exception:
        p_ee = np.ones(len(X_sc), dtype=int)

    votes = (p_if == -1).astype(int) + (p_lof == -1).astype(int) + (p_ee == -1).astype(int)
    majority = votes >= 2
    return times[majority].tolist()   # list of anomaly epoch START times (2s epochs)


# ══════════════════════════════════════════════════════════════════════════
#  3. Simulate the preprocessing sliding-window scheme and compute overlap
# ══════════════════════════════════════════════════════════════════════════

def simulate_windows(session_duration_s: float,
                     window_len: float = EEG_WINDOW_LEN,
                     step: float = EEG_STEP) -> list:
    """Reproduces the (offset_start, offset_end) pairs preprocessing would
    generate for a session of this duration, using a dense EEG_STEP=1s
    sliding window across the whole session (a reasonable approximation of
    the trial-level windowing without needing exact trial onset markers)."""
    windows = []
    t = 0.0
    while t + window_len <= session_duration_s:
        windows.append((t, t + window_len))
        t += step
    return windows


def merge_anomaly_intervals(anomaly_epoch_starts: list,
                            anomaly_epoch_len: float = 2.0) -> list:
    """
    BUG FIX: anomaly epochs are 2s long but only 1s apart (epoch_len=2.0,
    overlap=0.5 in epoch_signal). When several consecutive epochs are
    flagged, they represent one continuous anomaly stretch, not separate
    2s blocks. The previous version checked overlap against each 2s epoch
    individually, so even a long, clearly anomalous stretch (5-6 flagged
    epochs = ~6s continuous) never accumulated more than a single 2s
    epoch's overlap with any 10s window -- causing every removal_fraction
    to drop to exactly 0% for any threshold >= 0.30 (2s/10s), regardless
    of how much of the window was actually anomalous. FIX: merge
    consecutive/overlapping raw epoch intervals into continuous
    [start, end] anomaly intervals first, THEN compute overlap against
    those merged intervals.

    Returns
    -------
    list of (start, end) tuples -- merged, non-overlapping anomaly intervals
    """
    if not anomaly_epoch_starts:
        return []
    intervals = sorted([(s, s + anomaly_epoch_len) for s in anomaly_epoch_starts])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:   # overlapping or contiguous -> merge
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged


def compute_removal_fraction(windows: list, anomaly_epoch_starts: list,
                             anomaly_epoch_len: float,
                             overlap_threshold: float) -> tuple[int, int]:
    """
    For each simulated window, computes overlap with the MERGED anomaly
    intervals (see merge_anomaly_intervals -- this is the bug fix).
    Returns (n_removed, n_total).
    """
    merged_intervals = merge_anomaly_intervals(anomaly_epoch_starts, anomaly_epoch_len)
    n_removed = 0
    for w_start, w_end in windows:
        window_dur = w_end - w_start
        removed = False
        for a_start, a_end in merged_intervals:
            overlap = max(0.0, min(w_end, a_end) - max(w_start, a_start))
            if overlap / window_dur >= overlap_threshold:
                removed = True
                break
        if removed:
            n_removed += 1
    return n_removed, len(windows)


# ══════════════════════════════════════════════════════════════════════════
#  Main
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 78)
    print("  Epoch-Level Filter Calibration Test")
    print(f"  Subjects: {TEST_SUBJECTS} | Sessions: {SESSIONS}")
    print(f"  Simulated window scheme: {EEG_WINDOW_LEN}s window, {EEG_STEP}s step")
    print(f"  Candidate overlap thresholds: {CANDIDATE_THRESHOLDS}")
    print("=" * 78)

    rows = []   # for CSV output

    for subject_id in TEST_SUBJECTS:
        print(f"\n[{subject_id}]")
        for session in SESSIONS:
            data = load_session_eeg(subject_id, session)
            if data is None:
                print(f"  S{session}: [SKIP] data not found")
                continue

            session_dur = data.shape[1] / SFREQ
            print(f"  S{session}: {data.shape[0]}ch x {data.shape[1]} samples "
                  f"({session_dur:.0f}s) ... computing ML anomalies ...", end=" ", flush=True)

            anomaly_times = get_ml_anomaly_times(data, SFREQ)
            windows = simulate_windows(session_dur)

            print(f"{len(anomaly_times)} anomaly epochs, {len(windows)} simulated windows")

            for thresh in CANDIDATE_THRESHOLDS:
                n_removed, n_total = compute_removal_fraction(
                    windows, anomaly_times, anomaly_epoch_len=2.0,
                    overlap_threshold=thresh
                )
                frac = n_removed / n_total if n_total > 0 else 0.0
                rows.append({
                    "subject": subject_id, "session": session,
                    "overlap_threshold": thresh,
                    "n_windows_total": n_total,
                    "n_windows_removed": n_removed,
                    "fraction_removed": round(frac, 4),
                })
                flag = ("OK" if frac < 0.30 else
                        "BORDERLINE" if frac < 0.40 else
                        "TOO HIGH -- consider full-subject removal")
                print(f"    threshold={thresh:.2f}: "
                      f"{n_removed}/{n_total} windows removed "
                      f"({frac:.1%})  [{flag}]")

            del data

    # ── save CSV ──────────────────────────────────────────────────────────
    csv_path = os.path.join(OUTPUT_DIR, "epoch_filter_calibration.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["subject", "session", "overlap_threshold",
                                          "n_windows_total", "n_windows_removed",
                                          "fraction_removed"])
        w.writeheader()
        for r in rows:
            w.writerow(r)
    print(f"\n[CSV saved] {csv_path}")

    # ── per-subject recommendation at the proposed default threshold (0.30) ──
    print("\n" + "=" * 78)
    print("  RECOMMENDATION (at overlap_threshold=0.30, averaged across sessions)")
    print("=" * 78)
    default_thresh = 0.30
    for subject_id in TEST_SUBJECTS:
        subj_rows = [r for r in rows if r["subject"] == subject_id
                    and r["overlap_threshold"] == default_thresh]
        if not subj_rows:
            continue
        avg_frac = sum(r["fraction_removed"] for r in subj_rows) / len(subj_rows)
        verdict = ("USE EPOCH-LEVEL FILTER" if avg_frac < 0.30 else
                   "BORDERLINE -- inspect per-session breakdown" if avg_frac < 0.40 else
                   "CONSIDER FULL-SUBJECT REMOVAL instead of filtering")
        print(f"  {subject_id}: avg fraction removed = {avg_frac:.1%}  ->  {verdict}")
    print("=" * 78)


  Epoch-Level Filter Calibration Test
  Subjects: ['VP001', 'VP006', 'VP008'] | Sessions: [1, 2, 3]
  Simulated window scheme: 10.0s window, 1.0s step
  Candidate overlap thresholds: [0.1, 0.2, 0.3, 0.4, 0.5]

[VP001]
  S1: 28ch x 131644 samples (658s) ... computing ML anomalies ... 60 anomaly epochs, 649 simulated windows
    threshold=0.10: 304/649 windows removed (46.8%)  [TOO HIGH -- consider full-subject removal]
    threshold=0.20: 269/649 windows removed (41.4%)  [TOO HIGH -- consider full-subject removal]
    threshold=0.30: 133/649 windows removed (20.5%)  [OK]
    threshold=0.40: 44/649 windows removed (6.8%)  [OK]
    threshold=0.50: 30/649 windows removed (4.6%)  [OK]
  S2: 28ch x 131644 samples (658s) ... computing ML anomalies ... 64 anomaly epochs, 649 simulated windows
    threshold=0.10: 338/649 windows removed (52.1%)  [TOO HIGH -- consider full-subject removal]
    threshold=0.20: 300/649 windows removed (46.2%)  [TOO HIGH -- consider full-subject removal]
    thresh